In [ ]:
# simple_chat_client.py - TUZATILGAN VERSIYA
import sys
import socket
import threading
import time
from datetime import datetime
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
import base64

class LoginWindow(QDialog):
    """Kirish oynasi"""
    def __init__(self):
        super().__init__()
        self.setup_ui()
    
    def setup_ui(self):
        self.setWindowTitle("Chatga Kirish")
        self.setFixedSize(350, 250)
        
        layout = QVBoxLayout()
        layout.setSpacing(10)
        
        # Sarlavha
        title = QLabel("CHAT DASTURI")
        title.setStyleSheet("font-size: 18px; font-weight: bold; color: #333;")
        title.setAlignment(Qt.AlignCenter)
        layout.addWidget(title)
        
        layout.addSpacing(10)
        
        # Foydalanuvchi nomi
        layout.addWidget(QLabel("Foydalanuvchi nomi:"))
        self.username_input = QLineEdit()
        self.username_input.setPlaceholderText("Ismingizni kiriting")
        layout.addWidget(self.username_input)
        
        # Server manzili
        layout.addWidget(QLabel("Server IP:"))
        self.server_input = QLineEdit("127.0.0.1")
        layout.addWidget(self.server_input)
        
        # Port
        layout.addWidget(QLabel("Port:"))
        self.port_input = QLineEdit("9999")
        layout.addWidget(self.port_input)
        
        layout.addSpacing(10)
        
        # Tugma
        self.connect_btn = QPushButton("ULANISH")
        self.connect_btn.clicked.connect(self.try_connect)
        layout.addWidget(self.connect_btn)
        
        # Status
        self.status_label = QLabel("")
        self.status_label.setStyleSheet("color: #666;")
        layout.addWidget(self.status_label)
        
        self.setLayout(layout)
    
    def try_connect(self):
        """Serverni ulanishni sinash"""
        username = self.username_input.text().strip()
        server = self.server_input.text().strip()
        port = self.port_input.text().strip()
        
        if not username:
            QMessageBox.warning(self, "Diqqat", "Iltimos, foydalanuvchi nomini kiriting!")
            return
        
        if not server:
            server = "127.0.0.1"
        
        if not port:
            port = "9999"
        
        try:
            port = int(port)
        except:
            QMessageBox.warning(self, "Xato", "Port noto'g'ri formatda!")
            return
        
        # Test connection
        self.status_label.setText("Ulanmoqda...")
        self.connect_btn.setEnabled(False)
        
        # Threadda sinash
        self.connect_thread = threading.Thread(
            target=self.test_connection,
            args=(username, server, port)
        )
        self.connect_thread.daemon = True
        self.connect_thread.start()
    
    def test_connection(self, username, server, port):
        """Ulanishni sinash"""
        try:
            # Oddiy socket bilan test
            test_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            test_socket.settimeout(3)
            test_socket.connect((server, port))
            test_socket.close()
            
            # Muvaffaqiyatli
            QMetaObject.invokeMethod(
                self,
                "connection_success",
                Qt.QueuedConnection,
                Q_ARG(str, username),
                Q_ARG(str, server),
                Q_ARG(int, port)
            )
            
        except Exception as e:
            QMetaObject.invokeMethod(
                self,
                "connection_failed",
                Qt.QueuedConnection,
                Q_ARG(str, str(e))
            )
    
    @pyqtSlot(str, str, int)
    def connection_success(self, username, server, port):
        """Ulanish muvaffaqiyatli"""
        self.status_label.setText("✅ Ulandi!")
        self.connect_btn.setEnabled(True)
        
        # Chat oynasini ochish
        self.chat_window = ChatWindow(username, server, port)
        self.chat_window.show()
        self.accept()
    
    @pyqtSlot(str)
    def connection_failed(self, error):
        """Ulanish muvaffaqiyatsiz"""
        self.status_label.setText(f"❌ Xato: {error}")
        self.connect_btn.setEnabled(True)
        QMessageBox.warning(self, "Xato", f"Serverga ulanib bo'lmadi:\n{error}")

class ChatWindow(QMainWindow):
    """Asosiy chat oynasi"""
    def __init__(self, username, server_ip, port):
        super().__init__()
        self.username = username
        self.server_ip = server_ip
        self.port = port
        self.client = None
        self.connected = False
        
        self.setup_ui()
        self.connect_to_server()
    
    def setup_ui(self):
        self.setWindowTitle(f"Chat - {self.username}")
        self.setGeometry(100, 100, 600, 500)
        
        # Asosiy widget
        main_widget = QWidget()
        self.setCentralWidget(main_widget)
        layout = QVBoxLayout(main_widget)
        layout.setSpacing(5)
        
        # Status bar
        self.status_bar = QStatusBar()
        self.setStatusBar(self.status_bar)
        self.update_status("Ulanmoqda...", "#ff9900")
        
        # Xabarlar oynasi
        self.messages_area = QTextBrowser()
        self.messages_area.setStyleSheet("""
            QTextBrowser {
                font-family: Arial;
                font-size: 13px;
                background-color: white;
                border: 1px solid #ddd;
            }
        """)
        layout.addWidget(self.messages_area)
        
        # Xabar yuborish paneli
        input_layout = QHBoxLayout()
        
        self.message_input = QLineEdit()
        self.message_input.setPlaceholderText("Xabaringizni yozing va Enter bosing...")
        self.message_input.returnPressed.connect(self.send_message)
        self.message_input.setEnabled(False)
        input_layout.addWidget(self.message_input)
        
        self.send_btn = QPushButton("Yuborish")
        self.send_btn.clicked.connect(self.send_message)
        self.send_btn.setEnabled(False)
        input_layout.addWidget(self.send_btn)
        
        layout.addLayout(input_layout)
        
        # Boshlang'ich xabar
        self.add_system_message(f"Xush kelibsiz, {self.username}!")
        self.add_system_message(f"Server: {self.server_ip}:{self.port}")
    
    def connect_to_server(self):
        """Serverga ulanish"""
        try:
            self.client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.client.settimeout(10)
            self.client.connect((self.server_ip, self.port))
            
            # Usernameni yuborish
            self.client.send(self.username.encode('utf-8'))
            
            self.connected = True
            self.message_input.setEnabled(True)
            self.send_btn.setEnabled(True)
            self.update_status("✅ Serverga ulandi", "green")
            self.add_system_message("Serverga muvaffaqiyatli ulandi!")
            
            # Xabarlarni qabul qilish
            self.receive_thread = threading.Thread(target=self.receive_messages, daemon=True)
            self.receive_thread.start()
            
        except Exception as e:
            self.update_status(f"❌ Xato: {str(e)}", "red")
            self.add_system_message(f"Ulanish xatosi: {str(e)}")
            QMessageBox.warning(self, "Xato", f"Serverga ulanib bo'lmadi:\n{str(e)}")
    
    def update_status(self, text, color):
        """Statusni yangilash"""
        self.status_bar.showMessage(text)
        self.status_bar.setStyleSheet(f"color: {color}; font-weight: bold;")
    
    def send_message(self):
        """Xabar yuborish"""
        if not self.connected:
            QMessageBox.warning(self, "Xato", "Serverga ulanmagan!")
            return
        
        message = self.message_input.text().strip()
        if not message:
            return
        
        try:
            # O'z xabarini ko'rsatish
            self.add_own_message(message)
            
            # Shifrlash (base64)
            encrypted = base64.b64encode(message.encode()).decode()
            
            # Serverga yuborish
            self.client.send(encrypted.encode('utf-8'))
            
            # Inputni tozalash
            self.message_input.clear()
            
        except Exception as e:
            self.update_status(f"Yuborish xatosi: {str(e)}", "red")
    
    def receive_messages(self):
        """Xabarlarni qabul qilish"""
        while self.connected:
            try:
                data = self.client.recv(4096)
                if not data:
                    self.connected = False
                    QMetaObject.invokeMethod(
                        self,
                        "connection_lost",
                        Qt.QueuedConnection
                    )
                    break
                
                # Xabarni qabul qilish
                encrypted_msg = data.decode('utf-8', errors='ignore')
                QMetaObject.invokeMethod(
                    self,
                    "process_received_message",
                    Qt.QueuedConnection,
                    Q_ARG(str, encrypted_msg)
                )
                
            except socket.timeout:
                continue
            except Exception as e:
                if self.connected:
                    self.connected = False
                    QMetaObject.invokeMethod(
                        self,
                        "connection_lost",
                        Qt.QueuedConnection
                    )
                break
    
    @pyqtSlot()
    def connection_lost(self):
        """Uzilganligi haqida xabar"""
        self.update_status("❌ Server bilan aloqa uzildi", "red")
        self.message_input.setEnabled(False)
        self.send_btn.setEnabled(False)
        self.add_system_message("Server bilan aloqa uzildi!")
    
    @pyqtSlot(str)
    def process_received_message(self, encrypted_msg):
        """Qabul qilingan xabarni qayta ishlash"""
        try:
            # Deshifrlash
            decoded_bytes = base64.b64decode(encrypted_msg)
            message = decoded_bytes.decode('utf-8', errors='ignore')
            
            # Ko'rsatish
            self.add_other_message(message)
        except:
            # Agar deshifrlab bo'lmasa, oddiy ko'rsatish
            self.add_other_message(f"[Shifrlangan] {encrypted_msg[:50]}...")
    
    def add_system_message(self, message):
        """Tizim xabarini qo'shish"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        html = f"""
        <div style='text-align: center; margin: 5px; color: #666;'>
            <i>{timestamp}: {message}</i>
        </div>
        """
        self.messages_area.append(html)
    
    def add_own_message(self, message):
        """O'z xabarini qo'shish"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        html = f"""
        <div style='text-align: right; margin: 5px;'>
            <div style='background-color: #0084ff; color: white; 
                 padding: 8px 12px; border-radius: 15px 15px 0 15px;
                 display: inline-block; max-width: 70%; word-wrap: break-word;'>
                {message}
            </div>
            <div style='font-size: 11px; color: #666; text-align: right;'>
                <b>Siz</b> • {timestamp}
            </div>
        </div>
        """
        self.messages_area.append(html)
        self.scroll_to_bottom()
    
    def add_other_message(self, message):
        """Boshqaning xabarini qo'shish"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        html = f"""
        <div style='text-align: left; margin: 5px;'>
            <div style='background-color: #f1f0f0; color: #333; 
                 padding: 8px 12px; border-radius: 15px 15px 15px 0;
                 display: inline-block; max-width: 70%; word-wrap: break-word;'>
                {message}
            </div>
            <div style='font-size: 11px; color: #666;'>
                <b>Boshqa foydalanuvchi</b> • {timestamp}
            </div>
        </div>
        """
        self.messages_area.append(html)
        self.scroll_to_bottom()
    
    def scroll_to_bottom(self):
        """Pastga scroll qilish"""
        scrollbar = self.messages_area.verticalScrollBar()
        scrollbar.setValue(scrollbar.maximum())
    
    def closeEvent(self, event):
        """Dasturni yopish"""
        if self.client:
            try:
                self.client.close()
            except:
                pass
        event.accept()

def main():
    app = QApplication(sys.argv)
    
    # Login oynasi
    login = LoginWindow()
    if login.exec_():
        return app.exec_()
    
    return 0

if __name__ == "__main__":
    main()